<a href="https://colab.research.google.com/github/burrows99/melanoma-classification/blob/master/colab_runner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Melanoma Classification — Colab Runner

Clones (or updates) the repo, sets up `uv`, configures Kaggle credentials, runs baseline + 4 experiment presets, then downloads plots and saved models.

In [3]:
REPO_URL = "https://github.com/burrows99/melanoma-classification.git"
REPO_DIR = "melanoma-classification"

# Always start from /content to avoid nested clones
%cd /content

# Remove stale clone if present, then clone fresh
!rm -rf {REPO_DIR}
!git clone {REPO_URL}

%cd {REPO_DIR}
print("Working directory: ", end="")
!pwd

/content
Cloning into 'melanoma-classification'...
remote: Enumerating objects: 284, done.
remote: Counting objects: 100% (171/171), done.
remote: Compressing objects: 100% (113/113), done.
remote: Total 284 (delta 93), reused 124 (delta 56), pack-reused 113 (from 2)
Receiving objects: 100% (284/284), 11.72 MiB | 17.68 MiB/s, done.
Resolving deltas: 100% (145/145), done.
/content/melanoma-classification
Working directory: /content/melanoma-classification


## Step 2 — Install uv and sync dependencies

In [4]:
import os

# Install uv (idempotent — safe to run every time)
!curl -LsSf https://astral.sh/uv/install.sh | sh

# Add uv to PATH for this session
uv_bin = os.path.expanduser("~/.local/bin")
os.environ["PATH"] = uv_bin + os.pathsep + os.environ.get("PATH", "")

# Sync all dependencies from pyproject.toml
!uv sync
print("Dependencies synced.")

downloading uv 0.11.7 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 123 packages in 1ms
Installed 112 packages in 286ms
 + albucore==0.0.24
 + albumentations==2.0.8
 + annotated-doc==0.0.4
 + annotated-types==0.7.0
 + anyio==4.13.0
 + brotli==1.2.0
 + certifi==2026.2.25
 + charset-normalizer==3.4.7
 + click==8.3.2
 + cloudpickle==3.1.2
 + contourpy==1.3.3
 + cuda-bindings==13.2.0
 + cuda-pathfinder==1.5.2
 + cuda-toolkit==13.0.2
 + cycler==0.12.1
 + fastapi==0.135.3
 + filelock==3.25.2
 + fonttools==4.62.1
 + fsspec==2026.3.0
 + grad-cam==1.5.5
 + gradio==6.12.0
 + gradio-client==2.4.1
 + groovy==0.1.2
 + h11==0.16.0
 + hf-gradio==0.3.0
 + hf-xet==1.4.3
 + httpcore==1.0.9
 + httpx==0.28.1
 + huggingface-hub==1.10.1
 + idna==3.11
 + jinja2==3.1.6
 + joblib==1.5.3
 + kagglehub==1.0.0
 + kagglesdk==0.1.18
 + kiwisolver==1.5.0
 + llvmlite==0.47.0

## Step 3 — Configure Kaggle credentials

Set your Kaggle username and API key below. You can find these at https://www.kaggle.com/settings → API → Create New Token.

In [5]:
import os, json, pathlib

KAGGLE_USERNAME = "mrburrows"  # <-- replace
KAGGLE_KEY      = "KGAT_7ebf0cae77a8cd59de30d0e1a51f7dfc"   # <-- replace

# Write ~/.kaggle/kaggle.json (required by kagglehub)
kaggle_cfg = pathlib.Path.home() / ".kaggle" / "kaggle.json"
kaggle_cfg.parent.mkdir(parents=True, exist_ok=True)
kaggle_cfg.write_text(json.dumps({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}))
kaggle_cfg.chmod(0o600)

# Also export as environment variables (belt-and-suspenders)
os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
os.environ["KAGGLE_KEY"]      = KAGGLE_KEY

print("Kaggle credentials configured.")

Kaggle credentials configured.


## Step 4 — Run baseline + experiments

- **Baseline (A)** — Adam, no scheduler, focal γ=2.0
- **Experiment 1 (B)** — Adam + CosineAnnealing
- **Experiment 2 (C)** — AdamW + CosineAnnealing
- **Experiment 3 (D)** — AdamW + CosineAnnealing + focal γ=1.5
- **Experiment 4** — Image-only ablation (no metadata)

In [6]:
import os
# Colab sets MPLBACKEND=module://matplotlib_inline.backend_inline which is
# inherited by uv run subprocesses but not installed in the venv — force Agg.
os.environ["MPLBACKEND"] = "Agg"
print("MPLBACKEND set to Agg")


MPLBACKEND set to Agg


### 4.1 — Baseline (A) — Adam, no scheduler, focal γ=2.0

In [ ]:
!uv run main.py --train

Dataset split: 30118 train, 7530 validation samples.
Metadata preprocessor fitted. Output features: 14
20:42:12  model.metadata_melanoma_model  INFO  Creating EfficientNet-B0 model with 14 metadata features.
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100% 20.5M/20.5M [00:00<00:00, 241MB/s]
20:42:13  train  INFO  Epoch 1/20  |  LR: 0.0001
Training:  14% 130/941 [00:22<02:13,  6.06it/s]

### 4.2 — Experiment 1 (B) — Adam + CosineAnnealing

In [ ]:
!uv run main.py --train --experiment 1

### 4.3 — Experiment 2 (C) — AdamW + CosineAnnealing

In [ ]:
!uv run main.py --train --experiment 2

### 4.4 — Experiment 3 (D) — AdamW + CosineAnnealing + focal γ=1.5

In [ ]:
!uv run main.py --train --experiment 3

### 4.5 — Experiment 4 — Image-only ablation (no metadata)

In [ ]:
!uv run main.py --train --experiment 4